# Model Toolchain Validation — Exploratory Notebook

This notebook verifies that `demucs-infer==4.2.2` loads and runs inference locally on Python 3.13 before any production code is written.

Evidence is generated by running real Python code and inspecting output shapes, dtypes, and signatures directly.

## Step 1.5: Runtime import check (torch + torchaudio)

Confirm torch and torchaudio import cleanly with no runtime warnings.

In [ ]:
import warnings
import torch
import torchaudio

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    _ = torch.zeros(1)

print("torch.__version__:", torch.__version__)
print("torchaudio.__version__:", torchaudio.__version__)
print(f"warnings during smoke test: {len(caught)}")
for w in caught:
    print(f"  - {w.category.__name__}: {w.message}")

assert len(caught) == 0, f"Emitted {len(caught)} warnings on import"

**Observed output:**
```
torch.__version__: 2.14.0+cpu
torchaudio.__version__: 2.11.0+cpu
warnings during smoke test: 0
```

## Step 2: Discover demucs_infer.pretrained public API

In [ ]:
import demucs_infer.pretrained as pre
print("Public names:", [x for x in dir(pre) if not x.startswith('_')])

**Observed output:**
```
Public names: ['AnyModelRepo', 'BagOnlyRepo', 'DEFAULT_MODEL', 'GDriveRepo', 'HDemucs', 'LocalRepo', 'ModelLoadingError', 'ModelOnlyRepo', 'Path', 'REMOTE_ROOT', 'ROOT_URL', 'RemoteRepo', 'SOURCES', 'add_model_flags', 'bold', 'demucs_unittest', 'fatal', 'get_model', 'get_model_from_args', 'logger', 'logging', 'tp']
```

## Step 3: DEFAULT_MODEL and SOURCES

In [ ]:
from demucs_infer.pretrained import get_model, DEFAULT_MODEL, SOURCES
print("DEFAULT_MODEL:", DEFAULT_MODEL)
print("SOURCES:", SOURCES)

**Observed output:**
```
DEFAULT_MODEL: htdemucs
SOURCES: ['drums', 'bass', 'other', 'vocals']
```

## Step 4: Load demucs_unittest and inspect

In [ ]:
import inspect
model = get_model('demucs_unittest')
print("type:", type(model).__name__)
print("audio_channels:", model.audio_channels)
print("samplerate:", model.samplerate)
print("sources:", model.sources)
print("get_model signature:", inspect.signature(get_model))

**Observed output:**
```
type: HDemucs
audio_channels: 2
samplerate: 44100
sources: ['drums', 'bass', 'other', 'vocals']
get_model signature: (name: str, repo: Optional[pathlib._local.Path] = None)
```

## Step 5: apply_model signature

In [ ]:
from demucs_infer.apply import apply_model
print("apply_model signature:", inspect.signature(apply_model))

**Observed output:**
```
apply_model signature: (model: Union[demucs_infer.apply.BagOfModels, demucs_infer.demucs.Demucs, demucs_infer.hdemucs.HDemucs, demucs_infer.htdemucs.HTDemucs], mix: Union[torch.Tensor, demucs_infer.apply.TensorChunk], shifts: int = 1, split: bool = True, overlap: float = 0.25, transition_power: float = 1.0, progress: bool = False, device=None, num_workers: int = 0, segment: Optional[float] = None, pool=None, lock=None, callback: Optional[Callable[[dict], NoneType]] = None, callback_arg: Optional[dict] = None) -> torch.Tensor
```

## Step 6: Real inference on 1-second stereo dummy

In [ ]:
import numpy as np
import torch

sr = model.samplerate
dummy = np.random.randn(model.audio_channels, sr).astype(np.float32)
mix = torch.from_numpy(dummy).unsqueeze(0)
print("input shape:", tuple(mix.shape), "dtype:", mix.dtype)

out = apply_model(model, mix, progress=False)
print("output shape:", tuple(out.shape), "dtype:", out.dtype)

sources = out[0]
for i, name in enumerate(model.sources):
    stem = sources[i]
    print(f"  {name}: shape={tuple(stem.shape)}, dtype={stem.dtype}, "
          f"min={stem.min().item():.4f}, max={stem.max().item():.4f}")

**Observed output:**
```
input shape: (1, 2, 44100) dtype: torch.float32
output shape: (1, 4, 2, 44100) dtype: torch.float32
  drums: shape=(2, 44100), dtype=torch.float32, min=-0.2347, max=0.1415
  bass: shape=(2, 44100), dtype=torch.float32, min=-0.4053, max=0.0969
  other: shape=(2, 44100), dtype=torch.float32, min=-0.1434, max=0.1738
  vocals: shape=(2, 44100), dtype=torch.float32, min=-0.1916, max=0.2166
```

## Step 7: Length preservation across sizes

In [ ]:
for n in [1000, 22050, 44100, 50000]:
    m = torch.randn(1, 2, n)
    o = apply_model(model, m, progress=False)
    print(f"input={n} -> output={tuple(o.shape)}; len preserved: {o.shape[-1] == n}")

**Observed output:**
```
input=1000 -> output=(1, 4, 2, 1000); len preserved: True
input=22050 -> output=(1, 4, 2, 22050); len preserved: True
input=44100 -> output=(1, 4, 2, 44100); len preserved: True
input=50000 -> output=(1, 4, 2, 50000); len preserved: True
```